# Prompt Optimization with Gemini for a Healthcare Pharmacovigilance Use Case  
**Two tracks in one notebook:**  
1. **MLflow** for prompt experimentation, tracking, and selection  
2. **Google Vertex AI Prompt Optimizer** for system-instruction optimization

## Business scenario
We will use a **real healthcare review dataset** and frame the task as an **adverse-event triage** problem:

> Given a patient drug review, classify the side-effect risk into **LOW / MEDIUM / HIGH** for downstream safety triage.

This is a practical enterprise pattern for:
- patient support triage
- pharmacovigilance workflows
- escalation routing
- QA of clinical-support copilots

## What this notebook does
- Uses the **same dataset** in both sections
- Uses **Gemini** as the LLM
- Keeps the workflow straightforward and production-oriented
- Evaluates prompts with simple business metrics: **accuracy, macro-F1, and JSON-format compliance**

## Environment prerequisites

### For the MLflow section
You need a Gemini API key exposed as an environment variable, for example:

```bash
export GEMINI_API_KEY="your_key"
```

### For the Vertex AI section
You need a Google Cloud project with Vertex AI enabled and authentication configured, for example:

```bash
export GOOGLE_CLOUD_PROJECT="your-project-id"
export GOOGLE_CLOUD_LOCATION="us-central1"
gcloud auth application-default login
```

> Notes:
> - The **MLflow** section uses the Gemini Developer API style through `google-genai`.
> - The **Vertex AI** section uses **Vertex AI authentication/project context** for prompt optimization.
> - If your team already injects secrets from a backend or notebook environment, replace the `os.getenv(...)` calls accordingly.

In [ ]:
# If needed, run this cell once.
# Restart the kernel after install if your environment asks for it.

# %pip install -q -U #   "mlflow>=3.5.0" #   "google-genai>=1.7.0" #   "google-cloud-aiplatform>=1.114.0" #   pandas numpy scikit-learn tqdm requests

In [ ]:
import os
import json
import time
import random
import textwrap
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
VERTEX_GEMINI_MODEL = os.getenv("VERTEX_GEMINI_MODEL", "gemini-2.5-flash")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")
GOOGLE_CLOUD_LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "file:./mlruns")

print({
    "GEMINI_MODEL": GEMINI_MODEL,
    "VERTEX_GEMINI_MODEL": VERTEX_GEMINI_MODEL,
    "GOOGLE_CLOUD_PROJECT": GOOGLE_CLOUD_PROJECT,
    "GOOGLE_CLOUD_LOCATION": GOOGLE_CLOUD_LOCATION,
    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "HAS_GEMINI_API_KEY": bool(GEMINI_API_KEY),
})

## Load a real healthcare dataset

We will use the public **Drug Reviews (Druglib.com)** style dataset layout that includes:
- `condition`
- free-text review fields
- side-effect severity labels

We will convert the original side-effect labels into a business-friendly triage target:

- **LOW**: no or mild side effects
- **MEDIUM**: moderate side effects
- **HIGH**: severe or extremely severe side effects

This makes the prompt optimization objective concrete and business-relevant.

In [ ]:
# Public mirror with UCI-compatible drug review structure
TRAIN_URL = "https://raw.githubusercontent.com/izzykayu/ConsumerDrugReviewMachineLearning/master/data/drugLib/drugLibTrain_raw.tsv"
TEST_URL = "https://raw.githubusercontent.com/izzykayu/ConsumerDrugReviewMachineLearning/master/data/drugLib/drugLibTest_raw.tsv"

train_raw = pd.read_csv(TRAIN_URL, sep="\t")
test_raw = pd.read_csv(TEST_URL, sep="\t")
raw_df = pd.concat([train_raw, test_raw], ignore_index=True)

raw_df.head(3)

In [ ]:
SIDE_EFFECT_MAP = {
    "No Side Effects": "LOW",
    "Mild Side Effects": "LOW",
    "Moderate Side Effects": "MEDIUM",
    "Severe Side Effects": "HIGH",
    "Extremely Severe Side Effects": "HIGH",
}

def build_review_text(row: pd.Series) -> str:
    parts = []
    for col, prefix in [
        ("benefitsReview", "Benefits"),
        ("sideEffectsReview", "Side effects"),
        ("commentsReview", "Overall comment"),
    ]:
        val = str(row.get(col, "")).strip()
        if val and val.lower() != "nan":
            parts.append(f"{prefix}: {val}")
    return "\n".join(parts)

df = raw_df.copy()
df["triage_label"] = df["sideEffects"].map(SIDE_EFFECT_MAP)
df["review_text"] = df.apply(build_review_text, axis=1)

df = df.dropna(subset=["condition", "review_text", "triage_label"]).copy()
df = df[df["review_text"].str.len() > 40].copy()

# Keep a manageable, stratified working set
work_df, _ = train_test_split(
    df[["condition", "review_text", "triage_label"]].copy(),
    train_size=min(240, len(df)),
    stratify=df["triage_label"],
    random_state=SEED,
)

train_df, eval_df = train_test_split(
    work_df,
    test_size=0.35,
    stratify=work_df["triage_label"],
    random_state=SEED,
)

print("train shape:", train_df.shape)
print("eval shape:", eval_df.shape)
display(train_df["triage_label"].value_counts().to_frame("count"))
display(train_df.head(3))

## Define the enterprise task

The model must read a patient review and predict a triage class in JSON:

```json
{"label": "LOW|MEDIUM|HIGH", "rationale": "..."}
```

### Guardrails for the task
- Use only the evidence in the review
- Focus on side-effect severity, not overall drug satisfaction
- Keep the rationale short and clinically sensible
- Return valid JSON only

In [ ]:
BASE_SYSTEM_PROMPT = """
You are assisting a healthcare safety-operations team.

Task:
Classify the patient's side-effect risk in the drug review into exactly one label:
LOW, MEDIUM, or HIGH.

Label policy:
- LOW: no side effects, minimal discomfort, or only mild tolerable effects.
- MEDIUM: noticeable side effects that may require monitoring or follow-up, but do not sound dangerous.
- HIGH: severe, alarming, dangerous, highly disruptive, or potentially urgent side effects.

Instructions:
- Use only the review text and condition provided.
- Focus on side-effect severity, not whether the drug worked well overall.
- If the review contains both positive drug effectiveness and significant adverse effects, prioritize the side-effect severity.
- Return JSON only with exactly two keys:
  - label
  - rationale
- Keep rationale under 35 words.
""".strip()

USER_TEMPLATE = """
Condition: {condition}

Patient review:
{review_text}
""".strip()

ALLOWED_LABELS = ["LOW", "MEDIUM", "HIGH"]

In [ ]:
def build_messages(system_prompt: str, condition: str, review_text: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": USER_TEMPLATE.format(condition=condition, review_text=review_text)},
    ]

def extract_json_payload(text: str) -> Dict:
    text = (text or "").strip()
    if not text:
        return {}
    # Try direct JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try fenced JSON
    if "```" in text:
        chunks = text.split("```")
        for chunk in chunks:
            chunk = chunk.strip()
            if chunk.startswith("json"):
                chunk = chunk[4:].strip()
            try:
                return json.loads(chunk)
            except Exception:
                continue

    # Try first JSON-like span
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass
    return {}

def normalize_label(payload: Dict) -> str:
    label = str(payload.get("label", "")).upper().strip()
    if label in ALLOWED_LABELS:
        return label
    return "UNPARSEABLE"

# Part 1 — Prompt optimization with MLflow

This section uses **MLflow as the optimization backbone**:
- track prompt candidates
- log metrics and sample outputs
- compare runs
- select the best prompt for the task

This is a very practical enterprise workflow because many teams want:
- prompt lineage
- experiment history
- comparable metrics
- reproducibility
- traces of Gemini calls

Instead of starting with a complex optimizer, we keep it straightforward:
1. generate a few improved prompt candidates
2. evaluate them on the same labeled set
3. select the best-performing prompt

In [ ]:
import mlflow

try:
    from google import genai
except ImportError as e:
    raise ImportError("Install google-genai before running this notebook.") from e

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("gemini_healthcare_prompt_optimization")

# Optional Gemini autologging for traces
try:
    mlflow.gemini.autolog(silent=True)
except Exception as e:
    print("Gemini autolog could not be enabled in this environment:", e)

print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

def call_gemini_dev_api(messages: List[Dict[str, str]], model: str = GEMINI_MODEL, temperature: float = 0.2) -> str:
    if gemini_client is None:
        raise ValueError("GEMINI_API_KEY is not set.")

    system_text = ""
    user_text = ""
    for m in messages:
        if m["role"] == "system":
            system_text += m["content"] + "\n"
        elif m["role"] == "user":
            user_text += m["content"] + "\n"

    full_prompt = f"System instruction:\n{system_text.strip()}\n\nUser input:\n{user_text.strip()}"
    response = gemini_client.models.generate_content(
        model=model,
        contents=full_prompt,
    )
    return getattr(response, "text", "") or ""

def score_predictions(y_true: List[str], y_pred: List[str], raw_outputs: List[str]) -> Dict[str, float]:
    valid_mask = [p in ALLOWED_LABELS for p in y_pred]
    parse_rate = float(np.mean(valid_mask)) if len(valid_mask) else 0.0

    cleaned_pred = [p if p in ALLOWED_LABELS else "LOW" for p in y_pred]
    return {
        "accuracy": accuracy_score(y_true, cleaned_pred),
        "macro_f1": f1_score(y_true, cleaned_pred, average="macro"),
        "json_parse_rate": parse_rate,
    }

def evaluate_prompt_with_dev_api(system_prompt: str, data: pd.DataFrame, max_rows: int = None) -> Tuple[Dict[str, float], pd.DataFrame]:
    sample_df = data.copy()
    if max_rows:
        sample_df = sample_df.head(max_rows).copy()

    records = []
    y_true, y_pred, raw_outputs = [], [], []

    for _, row in sample_df.iterrows():
        messages = build_messages(system_prompt, row["condition"], row["review_text"])
        raw_text = call_gemini_dev_api(messages)
        payload = extract_json_payload(raw_text)
        pred_label = normalize_label(payload)

        y_true.append(row["triage_label"])
        y_pred.append(pred_label)
        raw_outputs.append(raw_text)

        records.append({
            "condition": row["condition"],
            "true_label": row["triage_label"],
            "pred_label": pred_label,
            "rationale": payload.get("rationale", ""),
            "raw_output": raw_text,
        })

    metrics = score_predictions(y_true, y_pred, raw_outputs)
    pred_df = pd.DataFrame(records)
    return metrics, pred_df

In [ ]:
PROMPT_OPTIMIZER_META_PROMPT = """
You are a senior healthcare AI prompt engineer.

Rewrite the following system prompt to improve classification accuracy for a healthcare adverse-event triage task.

Requirements:
- Preserve the label set: LOW, MEDIUM, HIGH
- Make instructions more explicit and harder to misinterpret
- Optimize for structured JSON output
- Keep it concise and enterprise-ready
- Return JSON only:
  {"candidates": ["prompt 1", "prompt 2", "prompt 3", "prompt 4"]}

Original prompt:
"""{base_prompt}"""
""".strip()

def generate_prompt_candidates(base_prompt: str, n_expected: int = 4) -> List[str]:
    raw_text = call_gemini_dev_api(
        [{"role": "user", "content": PROMPT_OPTIMIZER_META_PROMPT.format(base_prompt=base_prompt)}],
        temperature=0.4,
    )
    payload = extract_json_payload(raw_text)
    candidates = payload.get("candidates", [])
    candidates = [str(x).strip() for x in candidates if str(x).strip()]
    # Fallbacks if the model doesn't behave
    if not candidates:
        candidates = [
            base_prompt + "\nAlways prioritize explicit adverse symptom severity over medication success.",
            base_prompt + "\nWhen symptoms sound urgent, debilitating, or dangerous, classify as HIGH.",
            base_prompt + "\nReturn strict compact JSON with no markdown and no extra text.",
        ]
    # Deduplicate but preserve order
    deduped = []
    seen = set()
    for p in [base_prompt] + candidates:
        key = p.strip()
        if key not in seen:
            deduped.append(key)
            seen.add(key)
    return deduped[: max(2, n_expected + 1)]

prompt_candidates = generate_prompt_candidates(BASE_SYSTEM_PROMPT)
print("Number of candidate prompts:", len(prompt_candidates))
for i, p in enumerate(prompt_candidates, start=1):
    print("\n" + "="*20 + f" Prompt {i} " + "="*20)
    print(p[:700])

In [ ]:
if gemini_client is None:
    raise ValueError("Set GEMINI_API_KEY before running the MLflow section.")

summary_rows = []
all_run_outputs = {}

with mlflow.start_run(run_name="gemini_mlflow_prompt_optimization") as parent_run:
    mlflow.log_param("model", GEMINI_MODEL)
    mlflow.log_param("n_train_rows", len(train_df))
    mlflow.log_param("n_eval_rows", len(eval_df))
    mlflow.log_param("task", "healthcare_adverse_event_triage")

    for idx, candidate_prompt in enumerate(prompt_candidates, start=1):
        with mlflow.start_run(run_name=f"candidate_{idx}", nested=True):
            mlflow.log_param("candidate_index", idx)
            mlflow.log_text(candidate_prompt, f"candidate_prompt_{idx}.txt")

            metrics, pred_df = evaluate_prompt_with_dev_api(candidate_prompt, eval_df, max_rows=min(60, len(eval_df)))

            for k, v in metrics.items():
                mlflow.log_metric(k, float(v))

            mlflow.log_table(pred_df.head(25), f"candidate_{idx}_sample_predictions.json")

            summary_rows.append({
                "candidate_index": idx,
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
                "json_parse_rate": metrics["json_parse_rate"],
                "prompt_preview": candidate_prompt[:180].replace("\n", " "),
            })
            all_run_outputs[idx] = {
                "prompt": candidate_prompt,
                "predictions": pred_df,
                "metrics": metrics,
            }

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["macro_f1", "accuracy", "json_parse_rate"],
    ascending=False
).reset_index(drop=True)

display(summary_df)

best_candidate_idx = int(summary_df.iloc[0]["candidate_index"])
best_mlflow_prompt = all_run_outputs[best_candidate_idx]["prompt"]
best_mlflow_metrics = all_run_outputs[best_candidate_idx]["metrics"]

print("Best MLflow prompt candidate:", best_candidate_idx)
print(best_mlflow_metrics)

In [ ]:
print("Best prompt selected by MLflow-tracked evaluation:\n")
print(best_mlflow_prompt)

In [ ]:
best_mlflow_predictions = all_run_outputs[best_candidate_idx]["predictions"].copy()
display(best_mlflow_predictions.head(10))

print(
    classification_report(
        eval_df.head(min(60, len(eval_df)))["triage_label"],
        [p if p in ALLOWED_LABELS else "LOW" for p in best_mlflow_predictions["pred_label"]],
        digits=4,
    )
)

## Interpretation

At this point, MLflow has been used as the practical optimization loop:
- prompt candidates were generated
- each candidate was evaluated on the same labeled set
- metrics were logged
- the best prompt was selected

This pattern is often enough for enterprise teams that want:
- a prompt benchmark
- traceability
- a repeatable optimization routine
- compatibility with their existing Gemini application code

# Part 2 — Prompt optimization with Google Vertex AI

This section keeps the **same healthcare dataset and same task**, but uses **Vertex AI Prompt Optimizer**.

We will:
1. start from the same baseline prompt
2. run **Vertex zero-shot prompt optimization**
3. evaluate the optimized prompt on the same evaluation set
4. compare baseline vs optimized results

> Zero-shot optimizer is the most straightforward option and avoids the heavier setup of data-driven batch optimization.

In [ ]:
try:
    import vertexai
    from google import genai as google_genai
except ImportError as e:
    raise ImportError("Install google-cloud-aiplatform and google-genai to run the Vertex section.") from e

if not GOOGLE_CLOUD_PROJECT:
    raise ValueError("Set GOOGLE_CLOUD_PROJECT before running the Vertex section.")

vertex_client = vertexai.Client(
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
)

vertex_genai_client = google_genai.Client(
    vertexai=True,
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
)

print("Vertex clients initialized.")

In [ ]:
# Vertex zero-shot prompt optimization
vertex_optimization_response = vertex_client.prompt_optimizer.optimize_prompt(
    prompt=BASE_SYSTEM_PROMPT
)

vertex_optimized_prompt = None
if getattr(vertex_optimization_response, "parsed_response", None):
    parsed = vertex_optimization_response.parsed_response
    vertex_optimized_prompt = getattr(parsed, "suggested_prompt", None)

if not vertex_optimized_prompt:
    vertex_optimized_prompt = getattr(vertex_optimization_response, "raw_text_response", None)

print("Vertex optimized prompt:\n")
print(vertex_optimized_prompt[:2500] if vertex_optimized_prompt else "No optimized prompt returned.")

In [ ]:
def call_gemini_vertex(messages: List[Dict[str, str]], model: str = VERTEX_GEMINI_MODEL) -> str:
    system_text = ""
    user_text = ""
    for m in messages:
        if m["role"] == "system":
            system_text += m["content"] + "\n"
        elif m["role"] == "user":
            user_text += m["content"] + "\n"

    full_prompt = f"System instruction:\n{system_text.strip()}\n\nUser input:\n{user_text.strip()}"
    response = vertex_genai_client.models.generate_content(
        model=model,
        contents=full_prompt,
    )
    return getattr(response, "text", "") or ""

def evaluate_prompt_with_vertex(system_prompt: str, data: pd.DataFrame, max_rows: int = None) -> Tuple[Dict[str, float], pd.DataFrame]:
    sample_df = data.copy()
    if max_rows:
        sample_df = sample_df.head(max_rows).copy()

    records = []
    y_true, y_pred, raw_outputs = [], [], []

    for _, row in sample_df.iterrows():
        messages = build_messages(system_prompt, row["condition"], row["review_text"])
        raw_text = call_gemini_vertex(messages)
        payload = extract_json_payload(raw_text)
        pred_label = normalize_label(payload)

        y_true.append(row["triage_label"])
        y_pred.append(pred_label)
        raw_outputs.append(raw_text)

        records.append({
            "condition": row["condition"],
            "true_label": row["triage_label"],
            "pred_label": pred_label,
            "rationale": payload.get("rationale", ""),
            "raw_output": raw_text,
        })

    metrics = score_predictions(y_true, y_pred, raw_outputs)
    pred_df = pd.DataFrame(records)
    return metrics, pred_df

In [ ]:
vertex_baseline_metrics, vertex_baseline_preds = evaluate_prompt_with_vertex(
    BASE_SYSTEM_PROMPT,
    eval_df,
    max_rows=min(60, len(eval_df)),
)

vertex_optimized_metrics, vertex_optimized_preds = evaluate_prompt_with_vertex(
    vertex_optimized_prompt,
    eval_df,
    max_rows=min(60, len(eval_df)),
)

vertex_compare_df = pd.DataFrame([
    {"prompt_version": "baseline", **vertex_baseline_metrics},
    {"prompt_version": "vertex_optimized", **vertex_optimized_metrics},
]).sort_values(["macro_f1", "accuracy"], ascending=False)

display(vertex_compare_df)

In [ ]:
display(vertex_optimized_preds.head(10))

## Optional: skeleton for Vertex data-driven optimization

If your team wants a stronger optimizer than zero-shot, Vertex also supports **data-driven optimization**.
That route is more production-grade, but it requires extra setup such as a config file in GCS and the appropriate service account/project configuration.

The cell below is intentionally left as a skeleton for later extension.

In [ ]:
# Optional advanced path: Vertex data-driven optimizer (skeleton)
#
# from vertexai import types
#
# project_number = "YOUR_PROJECT_NUMBER"
# vapo_config = types.PromptOptimizerVAPOConfig(
#     config_path="gs://YOUR_BUCKET/prompt_optimizer_config.json",
#     service_account_project_number=project_number,
#     wait_for_completion=False,
# )
#
# result = vertex_client.prompt_optimizer.optimize(
#     method="vapo",
#     config=vapo_config,
# )
#
# result

# Final comparison and recommendation

In [ ]:
final_comparison = pd.DataFrame([
    {"approach": "MLflow best candidate", **best_mlflow_metrics},
    {"approach": "Vertex baseline", **vertex_baseline_metrics},
    {"approach": "Vertex optimized", **vertex_optimized_metrics},
]).sort_values(["macro_f1", "accuracy", "json_parse_rate"], ascending=False)

display(final_comparison)

## Recommended production next steps

### If you prefer MLflow as your optimization control plane
Use the MLflow route when you want:
- experiment tracking
- prompt lineage
- side-by-side run comparison
- easy integration into an existing LLMOps workflow
- flexibility to test many handcrafted or model-generated prompt candidates

### If you prefer a managed Google-native optimization path
Use the Vertex route when you want:
- Google-managed optimization workflows
- tighter integration with Vertex AI
- future extension into managed evaluation and data-driven prompt optimization

## How to adapt this notebook to your internal data
Replace the public dataset with your own table containing:
- `condition` or business context
- free-text input
- expected label

Then keep the same evaluation contract:
- same JSON schema
- same metric block
- same MLflow and Vertex comparison structure

## Source references used to shape this notebook

- MLflow prompt optimization and prompt evaluation docs
- MLflow Gemini tracing support
- Vertex AI Prompt Optimizer docs
- Vertex AI / Google Gen AI SDK docs
- UCI Drug Reviews dataset description

These sources were used to keep the notebook aligned to current product capabilities and current SDK direction.